# Match C-SPAN Programs to Hearings

| Item | Description |
|---|---|
| Goal | Link each congressional hearing to the C-SPAN program that covered it, so that a generated summary can be compared with an abstract written by a person about the same hearing. C-SPAN carries no GovInfo identifier, so the link is made on metadata alone: date and chamber to narrow the candidates, title words and witness surnames to score them, and mutual best match to accept. No summary text and no abstract text enter the key, so the comparison in T1.3.2 is not circular. |
| Input | `hearings_all.parquet` from D1, the 36,378 hearings; `programs.parquet` from D2, the 18,697 C-SPAN programs; `hearing_summaries.parquet` from T1.2, to flag the matches that have a summary. |
| Output | `Processed/cspan_matches.parquet` in the T1.3 scratch folder: 3,812 one-to-one matches with the hearing, the program, its abstract, the match score, and `in_sst`; 351 of them carry a generated summary. |
| Variables | • `score`: IDF-weighted cosine of the two titles plus half the share of witness surnames found in the abstract<br>• `in_sst`: the hearing is in the science committee sample and has a summary |
| Run | Last run 2026-08-21; the cell outputs shown are from that run. Not rerun: T1.3.2 and the proposal were built on this table. Rebuilt once on 2026-10-31 into a scratch copy and found identical. |

## Step 1. Import packages

In [ ]:
import html
import math
import re
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd

## Step 2. Define file locations

In [ ]:
PROJECT_FOLDER = Path.cwd().resolve().parents[2]  # <pipeline>/<section>/<subfolder>/ sits three levels below the clone
SCRATCH_FOLDER = Path("/scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis")  # the one line to change on another machine
NOTEBOOK_SUBFOLDER = Path("T - Three Step Text Analysis Pipeline/T1 - Text Summarization/T1.3 - Validation of Summaries")
NOTEBOOK_NAME = "T1.3.1 - Validation - Match C-SPAN Programs to Hearings"
PROCESSED_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Processed"
HEARINGS_FOLDER = SCRATCH_FOLDER / "D - Data Collection and Processing/D1 - GovInfo Congressional Hearings" / "Processed"
PROGRAMS_FOLDER = SCRATCH_FOLDER / "D - Data Collection and Processing/D2 - C-SPAN Reference Abstracts" / "Processed"
SUMMARIES_FOLDER = SCRATCH_FOLDER / "T - Three Step Text Analysis Pipeline/T1 - Text Summarization/T1.2 - Process Entire Sample Congressional Hearings" / "Processed"

PROCESSED_FOLDER.mkdir(parents=True, exist_ok=True)

## Step 3. Set the matching rules and load the three tables

A candidate program must air within a day of a sitting date and in the same chamber. A match is accepted when it is the best candidate for both sides, scores at least 0.55, and beats the runner-up by at least 0.05. Programs whose abstract has fewer than ten words are left out.

In [1]:
SEED = 20260821
rng = np.random.default_rng(SEED)

SCORE_MIN = 0.55  # match acceptance threshold
MARGIN_MIN = 0.05  # the best candidate must beat the runner-up by this
DAY_WINDOW = 1  # days around a sitting date
ABSTRACT_MIN_WORDS = 10

hearings = pd.read_parquet(HEARINGS_FOLDER / "hearings_all.parquet",
                           columns=["packageId", "title", "chamber", "year", "heldDate_first", "heldDates", "witnesses", "committee_ids", "committee_name"])
programs = pd.read_parquet(PROGRAMS_FOLDER / "programs.parquet", columns=["id", "title", "description", "date", "categories"])
summaries = pd.read_parquet(SUMMARIES_FOLDER / "hearing_summaries.parquet", columns=["packageId", "summary", "status", "year", "model"])

written_by = summaries.loc[summaries["status"].eq("ok"), "model"].unique()
assert len(written_by) == 1, f"summaries come from more than one model: {written_by}"
SUMMARY_MODEL = written_by[0]

print(f"hearings  {len(hearings):>7,}  {hearings['year'].min()}-{hearings['year'].max()}")
print(f"programs  {len(programs):>7,}")
print(f"summaries {len(summaries):>7,}  {int(summaries['status'].eq('ok').sum()):,} ok, written by {SUMMARY_MODEL}")

hearings   36,378  1995-2025
programs   18,697
summaries   2,352  2,352 ok, written by Qwen/Qwen3.5-4B


## Step 4. Score the candidate pairs and accept the mutual best matches

Title overlap is weighted by inverse document frequency. Shared words such as "fiscal year budget" then count for little, and a word such as "NASA" does the work. Each title vector is built from a sorted set of words, which keeps the floating-point sums identical from run to run. Unsorted, the same input accepted 3,805, 3,808, and 3,811 matches on three runs. Mutual best match makes the link one to one, and a program re-aired under a new id keeps one row.

In [2]:
CHAMBER_OF = {"House Committee": "HOUSE", "Senate Committee": "SENATE", "Joint Committee": "JOINT"}
TITLE_STOP = set("""a an the of on for and to in with by at from committee subcommittee
hearing hearings united states congress congressional session first second third
part day before joint full committees testimony testify testified witnesses examine""".split())
WITNESS_STOP = set("""chairman ranking member doctor honorable professor president director
secretary administrator commissioner committee states united washington university
department office general counsel deputy assistant associate executive institute
national american center school college research association corporation company""".split())


def clean_abstract(text):
    return re.sub(r"\s+", " ", re.sub(r"<[^>]+>", " ", html.unescape(str(text or "")))).strip()


def chambers_of(categories):
    return {CHAMBER_OF[p] for p in str(categories).split("|") if p in CHAMBER_OF}


def title_tokens(text):
    return [w for w in re.findall(r"[a-z0-9']+", str(text).lower()) if w not in TITLE_STOP and len(w) > 2]


def witness_surnames(witnesses):
    names = list(witnesses) if witnesses is not None else []
    return {p.lower() for w in names for p in re.findall(r"[A-Z][a-zA-Z'\-]{3,}", str(w))} - WITNESS_STOP


def abstract_words(text):
    return set(re.findall(r"[a-z'\-]{4,}", text.lower()))


# the reference side: clean text, Eastern dates, chamber, usable abstracts only
programs["abstract"] = programs["description"].map(clean_abstract)
programs["aired"] = pd.to_datetime(programs["date"], utc=True).dt.tz_convert("US/Eastern").dt.strftime("%Y-%m-%d")
programs["chambers"] = programs["categories"].map(chambers_of)
keep = programs["abstract"].str.split().str.len() >= ABSTRACT_MIN_WORDS
print(f"{int(keep.sum()):,} programs carry a usable abstract ({keep.mean():.1%})")
programs = programs.loc[keep].reset_index(drop=True)

hearing_tokens = hearings["title"].map(title_tokens)
program_tokens = programs["title"].map(title_tokens)
document_frequency = Counter()
for tokens in list(hearing_tokens) + list(program_tokens):
    document_frequency.update(set(tokens))
N = len(hearing_tokens) + len(program_tokens)
IDF = {w: math.log(N / (1 + n)) for w, n in document_frequency.items()}


def unit_vector(tokens):
    """IDF vector of a title, L2-normalized. sorted() keeps the run reproducible."""
    v = {w: IDF.get(w, math.log(N)) for w in sorted(set(tokens))}
    norm = math.sqrt(sum(x * x for x in v.values())) or 1.0
    return {w: x / norm for w, x in v.items()}


hearing_vec = [unit_vector(t) for t in hearing_tokens]
program_vec = [unit_vector(t) for t in program_tokens]
hearing_names = hearings["witnesses"].map(witness_surnames).tolist()
program_words = programs["abstract"].map(abstract_words).tolist()
program_chambers = programs["chambers"].tolist()

by_day = defaultdict(list)
for j, day in enumerate(programs["aired"].values):
    by_day[day].append(j)

# score every candidate pair: same chamber, within a day of a sitting date
rows = []
for i, (held, first, chamber) in enumerate(zip(hearings["heldDates"], hearings["heldDate_first"], hearings["chamber"])):
    left, names = hearing_vec[i], hearing_names[i]
    if not left:
        continue
    dates = [d for d in (list(held) if held is not None else []) if pd.notna(d)]
    if not dates and pd.notna(first):
        dates = [first]
    days = {(pd.Timestamp(d) + pd.Timedelta(days=k)).strftime("%Y-%m-%d") for d in dates for k in range(-DAY_WINDOW, DAY_WINDOW + 1)}
    for day in days:
        for j in by_day.get(day, ()):
            chambers = program_chambers[j]
            if chamber and chambers and chamber != "JOINT" and chamber not in chambers and "JOINT" not in chambers:
                continue
            right = program_vec[j]
            if not right:
                continue
            cos = sum(w * right[t] for t, w in left.items() if t in right)
            wit = len(names & program_words[j]) / len(names) if names else 0.0
            rows.append((i, j, cos, wit, cos + 0.5 * wit))

candidates = pd.DataFrame(rows, columns=["hi", "pj", "cos", "wit", "score"])

# accept: mutual best, over the threshold, clear of the runner-up
ranked = candidates.sort_values("score", ascending=False, kind="mergesort")
best = ranked.drop_duplicates("hi").set_index("hi")
runner_up = (ranked.merge(best["pj"].rename("top"), left_on="hi", right_index=True).query("pj != top").groupby("hi")["score"].max())
best["margin"] = best["score"] - runner_up.reindex(best.index).fillna(0.0)
best["mutual"] = (ranked.drop_duplicates("pj").set_index("pj")["hi"].reindex(best["pj"]).values == best.index)
accepted = best.loc[best["mutual"] & (best["score"] >= SCORE_MIN) & (best["margin"] >= MARGIN_MIN)]

matches = pd.DataFrame({
    "packageId": hearings["packageId"].values[accepted.index],
    "hearing_title": hearings["title"].values[accepted.index],
    "heldDate_first": hearings["heldDate_first"].values[accepted.index],
    "year": hearings["year"].values[accepted.index],
    "committee_ids": hearings["committee_ids"].values[accepted.index],
    "committee_name": hearings["committee_name"].values[accepted.index],
    "cspan_id": programs["id"].values[accepted["pj"].values],
    "cspan_title": programs["title"].values[accepted["pj"].values],
    "abstract": programs["abstract"].values[accepted["pj"].values],
    "score": accepted["score"].values,
})
# C-SPAN re-airs the same program under a new id; one abstract, one row
matches["_text"] = matches["abstract"].str.lower().str.replace(r"[^a-z]", "", regex=True).str[:400]
matches = (matches.sort_values("score", ascending=False, kind="mergesort").drop_duplicates("_text")
           .drop(columns="_text").sort_values("packageId").reset_index(drop=True))
matches["in_sst"] = matches["packageId"].isin(set(summaries["packageId"]))
matches.to_parquet(PROCESSED_FOLDER / "cspan_matches.parquet", compression="zstd", index=False)

assert not matches["packageId"].duplicated().any()
assert not matches["cspan_id"].duplicated().any()
print(f"{len(candidates):,} candidate pairs -> {len(matches):,} accepted matches, one-to-one, across "
      f"{len(set().union(*matches['committee_ids'].map(set))):,} committees")
print(f"  with a generated summary: {int(matches['in_sst'].sum()):,}")
print(f"  wrote {PROCESSED_FOLDER / 'cspan_matches.parquet'}\n")

print("ten accepted matches at random:")
for r in matches.iloc[rng.choice(len(matches), 10, replace=False)].itertuples():
    print(f"  {r.heldDate_first}  {r.hearing_title[:80]}")
    print(f"              {r.cspan_title[:80]}")

17,373 programs carry a usable abstract (92.9%)


243,495 candidate pairs -> 3,812 accepted matches, one-to-one, across 47 committees
  with a generated summary: 351
  wrote /project2/njia_1089/acroh/projects/llm_method/data/cleaned/01_text_summarization/cspan_matches.parquet

ten accepted matches at random:
  2009-07-22  ARBITRATION OR ARBITRARY: THE MISUSE OF MANDATORY ARBITRATION TO COLLECT CONSUME
              Arbitration Or Arbitrary: The Misuse Of Mandatory Arbitration To Collect Consume
  2017-07-13  EXAMINING VA'S PROCESSING OF GULF WAR ILLNESS CLAIMS
              Gulf War Illness Claims
  2023-05-23  THE BIDEN BORDER CRISIS: PART III
              Hearing on the Biden Administration's Immigration and Border Policies
  2012-08-01  BREACH OF TRUST: ADDRESSING MISCONDUCT AMONG TSA SCREENERS
              Transportation Security Administration  Screeners
  2003-10-02  CHALLENGES FOR U.S. POLICY TOWARD CUBA
              U.S. Policy Toward Cuba
  2010-05-05  [H.A.S.C. No. 111-164]DEVELOPMENTS IN SECURITY AND STABILITY IN AFGHANI

## Step 5. Check the matches

- One row per hearing and one per program.
- Every accepted match clears the score and the margin.
- The matches with a summary are counted, and the agreement of the two titles is shown on ten random rows above.

In [ ]:
back = pd.read_parquet(PROCESSED_FOLDER / "cspan_matches.parquet")

assert not back["packageId"].duplicated().any(), "a hearing matched twice"
assert not back["cspan_id"].duplicated().any(), "a program matched twice"
print(f"1) {len(back):,} matches, one per hearing and one per program")

assert (back["score"] >= SCORE_MIN).all(), "a match is under the score threshold"
print(f"2) every match scores at least {SCORE_MIN}; median {back['score'].median():.2f}")

print(f"3) {int(back['in_sst'].sum()):,} matched hearings carry a generated summary; "
      f"they are the focal side of T1.3.2, and all {len(back):,} abstracts are its comparison pool")
print(f"   years covered: {back['year'].min()}-{back['year'].max()}")